# Notebook - create data

## libraries

In [38]:
#%pip install faker psycopg2-binary pandas

In [67]:
from faker import Faker
import psycopg
import random
import pandas as pd
from datetime import datetime

fake = Faker("en_US")

# Same data
Faker.seed(42)
random.seed(42)

## Database connection

In [68]:
conn = psycopg.connect(
    host="localhost",
    port=5433,
    dbname="ecommerce",
    user="data_engineer",
    password="data_engineer_dev"
)

cursor = conn.cursor()

print("Successful connection!")

Successful connection!


## Categories

In [41]:
categories = [
    "Electronics",
    "Computers",
    "Smartphones",
    "Home Appliances",
    "Furniture",
    "Clothing",
    "Footwear",
    "Sports",
    "Beauty",
    "Books",
    "Gaming",
    "Accessories",
    "Office Supplies",
    "Toys",
    "Kitchen",
    "Garden",
    "Automotive",
    "Pet Supplies",
    "Health",
    "Music"
]

In [42]:
category_ids = []

for category in categories:
    cursor.execute(
        """
        INSERT INTO categories (name)
        VALUES (%s)
        RETURNING category_id
        """,
        (category,)
    )

    category_id = cursor.fetchone()[0]
    category_ids.append(category_id)

conn.commit()

print(f"{len(category_ids)} categories inserted successfully!")

20 categories inserted successfully!


## Customers

In [43]:
customer_ids = []

for _ in range(10000):

    first_name = fake.first_name()
    last_name = fake.last_name()
    email = fake.unique.email()

    cursor.execute(
        """
        INSERT INTO customers (
            first_name,
            last_name,
            email
        )
        VALUES (%s, %s, %s)
        RETURNING customer_id
        """,
        (
            first_name,
            last_name,
            email
        )
    )

    customer_id = cursor.fetchone()[0]
    customer_ids.append(customer_id)

conn.commit()

print(f"{len(customer_ids)} customers inserted successfully!")

10000 customers inserted successfully!


## Products

In [44]:
product_ids = []

for _ in range(1000):

    category_id = random.choice(category_ids)

    product_name = f"{fake.word().title()} {random.choice([
        'Laptop',
        'Smartphone',
        'Headphones',
        'Monitor',
        'Keyboard',
        'Mouse',
        'Camera',
        'Tablet',
        'Speaker',
        'Chair',
        'Backpack',
        'Watch',
        'Printer',
        'Desk'
    ])}"

    description = fake.paragraph(nb_sentences=3)

    price = round(random.uniform(10, 5000), 2)

    stock_quantity = random.randint(0, 500)

    cursor.execute(
        """
        INSERT INTO products (
            name,
            description,
            price,
            stock_quantity,
            category_id
        )
        VALUES (%s, %s, %s, %s, %s)
        RETURNING product_id
        """,
        (
            product_name,
            description,
            price,
            stock_quantity,
            category_id
        )
    )

    product_id = cursor.fetchone()[0]
    product_ids.append(product_id)

conn.commit()

print(f"{len(product_ids)} products inserted successfully!")

1000 products inserted successfully!


## Orders

In [45]:
cursor.execute("""
    SELECT customer_id
    FROM customers
    ORDER BY customer_id
""")

customer_ids = [
    row[0]
    for row in cursor.fetchall()
]

print(f"{len(customer_ids)} customer IDs loaded.")

10000 customer IDs loaded.


In [46]:
order_ids = []

order_statuses = [
    "pending",
    "paid",
    "processing",
    "shipped",
    "delivered",
    "cancelled"
]

for _ in range(30000):

    customer_id = random.choice(customer_ids)

    order_date = fake.date_time_between(
        start_date="-2y",
        end_date="now"
    )

    status = random.choice(order_statuses)

    cursor.execute(
        """
        INSERT INTO orders (
            customer_id,
            order_date,
            status,
            total_amount
        )
        VALUES (%s, %s, %s, %s)
        RETURNING order_id
        """,
        (
            customer_id,
            order_date,
            status,
            0
        )
    )

    order_id = cursor.fetchone()[0]
    order_ids.append(order_id)

conn.commit()

print(f"{len(order_ids)} orders inserted successfully!")

30000 orders inserted successfully!


## Order items

In [48]:
cursor.execute("""
    SELECT product_id, price
    FROM products
    ORDER BY product_id
""")

product_prices = {
    row[0]: row[1]
    for row in cursor.fetchall()
}

product_ids = list(product_prices.keys())


cursor.execute("""
    SELECT order_id
    FROM orders
    ORDER BY order_id
""")

order_ids = [
    row[0]
    for row in cursor.fetchall()
]

conn.commit()


print(f"Products loaded: {len(product_ids):,}")
print(f"Orders loaded: {len(order_ids):,}")

if len(product_ids) == 0:
    raise ValueError(
        "No products found. Run the Products cell before Order Items."
    )

if len(order_ids) == 0:
    raise ValueError(
        "No orders found. Run the Orders cell before Order Items."
    )


order_items_count = 0

for order_id in order_ids:

    number_of_items = random.randint(1, 5)

    selected_products = random.sample(
        product_ids,
        number_of_items
    )

    total_amount = 0

    for product_id in selected_products:

        quantity = random.randint(1, 5)

        unit_price = product_prices[product_id]

        item_total = unit_price * quantity

        total_amount += item_total

        cursor.execute(
            """
            INSERT INTO order_items (
                order_id,
                product_id,
                quantity,
                unit_price
            )
            VALUES (%s, %s, %s, %s)
            """,
            (
                order_id,
                product_id,
                quantity,
                unit_price
            )
        )

        order_items_count += 1

    cursor.execute(
        """
        UPDATE orders
        SET total_amount = %s
        WHERE order_id = %s
        """,
        (
            total_amount,
            order_id
        )
    )


# Save all changes
conn.commit()

print(f"{order_items_count:,} order items inserted successfully!")

Products loaded: 1,000
Orders loaded: 30,000
89,792 order items inserted successfully!


## Payments

In [49]:
payment_methods = [
    "Credit Card",
    "Debit Card",
    "PayPal",
    "Bank Transfer",
    "Digital Wallet"
]

payment_statuses = [
    "Pending",
    "Approved",
    "Failed",
    "Refunded"
]

In [50]:
for order_id in order_ids:

    cursor.execute(
        """
        SELECT total_amount
        FROM orders
        WHERE order_id = %s
        """,
        (order_id,)
    )

    amount = cursor.fetchone()[0]

    cursor.execute(
        """
        INSERT INTO payments (
            order_id,
            payment_method,
            payment_status,
            amount
        )
        VALUES (%s, %s, %s, %s)
        """,
        (
            order_id,
            random.choice(payment_methods),
            random.choice(payment_statuses),
            amount
        )
    )

conn.commit()

print("Payments inserted successfully!")

Payments inserted successfully!


## Inventory

In [51]:
for product_id in product_ids:

    quantity = random.randint(0, 500)

    cursor.execute(
        """
        INSERT INTO inventory (
            product_id,
            quantity
        )
        VALUES (%s, %s)
        """,
        (
            product_id,
            quantity
        )
    )

conn.commit()

print("Inventory data inserted successfully!")

Inventory data inserted successfully!


## Data Quality

In [52]:
# Customers

cursor.execute("""
    SELECT COUNT(*)
    FROM customers
""")

print("Customers:", cursor.fetchone()[0])

Customers: 10000


In [53]:
# Products

cursor.execute("""
    SELECT COUNT(*)
    FROM products
""")

print("Products:", cursor.fetchone()[0])

Products: 1000


In [54]:
# Order

cursor.execute("""
    SELECT COUNT(*)
    FROM orders
""")

print("Orders:", cursor.fetchone()[0])

Orders: 30000


In [55]:
# Order Item

cursor.execute("""
    SELECT COUNT(*)
    FROM order_items
""")

print("Order Items:", cursor.fetchone()[0])

Order Items: 90413


In [56]:
# Payments

cursor.execute("""
    SELECT COUNT(*)
    FROM payments
""")

print("Payments:", cursor.fetchone()[0])

Payments: 30000


In [57]:
# Inventory

cursor.execute("""
    SELECT COUNT(*)
    FROM inventory
""")

print("Inventory:", cursor.fetchone()[0])

Inventory: 1000


## Check FKs

### Product -> Categories

In [58]:
cursor.execute("""
    SELECT COUNT(*)
    FROM products p
    LEFT JOIN categories c
        ON p.category_id = c.category_id
    WHERE c.category_id IS NULL
""")

print(
    "Products without category:",
    cursor.fetchone()[0]
)

Products without category: 0


### Orders → Customers

In [59]:
cursor.execute("""
    SELECT COUNT(*)
    FROM orders o
    LEFT JOIN customers c
        ON o.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
""")

print(
    "Orders without customer:",
    cursor.fetchone()[0]
)

Orders without customer: 0


### Order Items → Orders

In [60]:
cursor.execute("""
    SELECT COUNT(*)
    FROM order_items oi
    LEFT JOIN orders o
        ON oi.order_id = o.order_id
    WHERE o.order_id IS NULL
""")

print(
    "Order items without order:",
    cursor.fetchone()[0]
)

Order items without order: 0


### Order Items → Products

In [61]:
cursor.execute("""
    SELECT COUNT(*)
    FROM order_items oi
    LEFT JOIN products p
        ON oi.product_id = p.product_id
    WHERE p.product_id IS NULL
""")

print(
    "Order items without product:",
    cursor.fetchone()[0]
)

Order items without product: 0


## Count rows

In [69]:
tables = [
    "customers",
    "categories",
    "products",
    "orders",
    "order_items",
    "payments",
    "inventory"
]

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table}: {count:,}")

customers: 10,000
categories: 20
products: 1,000
orders: 30,000
order_items: 90,413
payments: 30,000
inventory: 1,000


## Close Conection

In [70]:
cursor.close()
conn.close()

print("PostgreSQL connection closed.")

PostgreSQL connection closed.
